In [44]:
%load_ext autoreload
%autoreload 2

from pydantic import BaseModel, Field, create_model
from beartype import beartype

import numpy as np
import yaml

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score

import optuna
import mne

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import sys
from pathlib import Path

# Находим корень проекта (поднимаемся на уровень выше папки notebooks)
root = Path.cwd().parent
sys.path.append(str(root))

from utils.TFRDataset import TFRDataset

from utils.normalisation import normalize_tfr_robust
from utils.AlexNet import AlexNetTFR
from lib.optuna_objective_makers import *
from utils.optuna_constraints import slope_constraint

from collections import Counter

from optuna.trial import TrialState
from utils.optuna_study_analyzers import pareto_front, feasible_trials_less_zero
import gc

from utils.train_eval_helpers import train_one_epoch, eval_one_epoch_f1_macro

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, timesteps, dropout):
        super(PositionalEncoding, self).__init__()
        self.dropout = torch.nn.Dropout(p=dropout)

        position = torch.arange(0, timesteps, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(timesteps, d_model)
        pe[:, 0::2] = 1*torch.sin(position * div_term)
        pe[:, 1::2] = 1*torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class Transformer(torch.nn.Module):
    def __init__(self, seq_len, embed_dim, nhead, dim_fc, num_layers, dropout, device = device):
        super(Transformer, self).__init__()
        self.embed_layer = nn.LazyLinear(embed_dim)
        self.position_encoder = PositionalEncoding(d_model=embed_dim, timesteps=seq_len, dropout=dropout)
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead,
                                                         dim_feedforward=dim_fc, dropout=dropout, batch_first=True)
        self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm_before_PE = torch.nn.LayerNorm(embed_dim)
        self.norm_after_PE = torch.nn.LayerNorm(embed_dim)
        self.fc1 = torch.nn.Linear(embed_dim, embed_dim)
        self.fc2 = torch.nn.Linear(embed_dim, embed_dim)
        # self.fc11 = torch.nn.Linear(embed_dim, embed_dim)
        # self.fc22 = torch.nn.Linear(embed_dim, embed_dim)
        self.fc3 = torch.nn.Linear(embed_dim, embed_dim)
        self.fc4 = torch.nn.Linear(embed_dim, embed_dim // 2)
        self.fc5 = torch.nn.Linear(embed_dim // 2, embed_dim // 4)
        self.fc6 = torch.nn.Linear(embed_dim // 4, 2)
        self.dropout = torch.nn.Dropout(dropout)
        self.activation = torch.nn.GELU()
        self.norm1 = torch.nn.LayerNorm(embed_dim)
        self.norm2 = torch.nn.LayerNorm(embed_dim)
        # self.norm11 = torch.nn.LayerNorm(embed_dim)
        # self.norm22 = torch.nn.LayerNorm(embed_dim)
        self.norm_conv = torch.nn.LayerNorm(embed_dim)
        # self.norm3 = torch.nn.LayerNorm(2)
        self.conv_layers = torch.nn.Sequential(
            # First Conv1d: input channels = embed_dim, output channels = embed_dim
            torch.nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            torch.nn.BatchNorm1d(embed_dim),  # BatchNorm on channel dimension
            torch.nn.ReLU(),
            # torch.nn.BatchNorm1d(embed_dim),
            
            # Second Conv1d: maintaining embed_dim channels
            # torch.nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            # torch.nn.BatchNorm1d(embed_dim),
            # torch.nn.ReLU(),
        )
    
        
    def forward(self, x):
        out = self.embed_layer(x)
        out = self.norm_before_PE(out)
        out = self.position_encoder(out)
        # residue = out
        # out = self.norm_before_PE(out)
        out = self.encoder(out)
        out = self.norm_after_PE(out)
        
#         out_conv = out.transpose(1, 2)  # -> [batch_size, embed_dim, seq_len]
#         out_conv = self.conv_layers(out_conv)  # -> [batch_size, embed_dim, seq_len]
#         out_conv = out_conv.transpose(1, 2) 
        
#         out = self.norm_conv(out_conv)#+out
        
        # out = self.norm_after_PE(out)
        residue = out
        out = self.dropout((self.activation(self.fc1(out))))
        
        out = self.norm1(out+residue)
        
        residue = out
        out = self.dropout((self.activation(self.fc2(out))))
        
        out = self.norm2(out+residue)
        
        out = self.dropout((self.activation(self.fc3(out))))
        # # out = out+residue
        out = self.dropout((self.activation(self.fc4(out))))
        out = self.dropout((self.activation(self.fc5(out))))
        out = (self.fc6(out))
        return out

In [59]:
# =========================================================
# 1. pooling по времени
# =========================================================

class SeqPool(nn.Module):

    def __init__(self, mode="mean"):
        super().__init__()
        assert mode in ["mean", "max", "last", "softmax", "none"], f"Unknown mode {mode}"
        self.mode = mode
        if self.mode == "softmax":
            self.score = nn.LazyLinear(1)

    def forward(self, x):
        # x: [B,T,D]

        if self.mode == "mean":
            return x.mean(dim=1)

        if self.mode == "max":
            return x.max(dim=1).values

        if self.mode == "last":
            return x[:, -1]

        if self.mode == "softmax":
            # x: [B,T,D]
            w = self.score(x)        # [B,T,1]
            w = torch.softmax(w, dim=1)
            return (x * w).sum(dim=1)   # [B,D]
        
        if self.mode == "none":
            return x

In [4]:
# =========================================================
# 2. бейзлайн reshape
# =========================================================

class TFRToSeqFlatten(nn.Module):
    """
    [B, C, F, T] -> [B, T, C*F]
    """
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(f"Expected [B,C,F,T], got {tuple(x.shape)}")

        B, C, Freq, Time = x.shape
        x = x.permute(0, 3, 1, 2).contiguous()   # [B,T,C,F]
        x = x.reshape(B, Time, C * Freq)         # [B,T,C*F]
        return x


# =========================================================
# 3. вариант A: свёртка по каналам
#    один вес на один канал
#    [B,C,F,T] -> [B,T,F]
# =========================================================

class TFRToSeqChannelConvCollapse(nn.Module):
    """
    Свёртка в домене каналов:
    - один вес на один канал
    - схлопывание вдоль каналов
    - частоты остаются фичами
    - время остаётся sequence

    Реализация через conv3d с kernel = [C,1,1].
    """
    def __init__(self, bias: bool = True):
        super().__init__()
        self.bias_enabled = bias

        self._built = False
        self._channels = None

        self.channel_kernel = None
        self.channel_bias = None

    def _build(self, channels: int, device, dtype):
        self._channels = channels

        self.channel_kernel = nn.Parameter(
            torch.empty(channels, device=device, dtype=dtype)
        )
        nn.init.kaiming_uniform_(self.channel_kernel.view(1, 1, channels), a=math.sqrt(5))

        if self.bias_enabled:
            self.channel_bias = nn.Parameter(
                torch.zeros(1, device=device, dtype=dtype)
            )
        else:
            self.register_parameter("channel_bias", None)

        self._built = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(f"Expected [B,C,F,T], got {tuple(x.shape)}")

        B, C, Freq, Time = x.shape

        if not self._built:
            self._build(C, x.device, x.dtype)

        if C != self._channels:
            raise ValueError(
                f"Channel count changed inside one model instance: expected C={self._channels}, got C={C}"
            )

        # [B,C,F,T] -> [B,1,C,F,T]
        x5 = x.unsqueeze(1)

        # kernel: [out_ch=1, in_ch=1, kC=C, kF=1, kT=1]
        kernel = self.channel_kernel.view(1, 1, C, 1, 1)

        out = F.conv3d(
            x5,
            weight=kernel,
            bias=self.channel_bias,
            stride=1,
            padding=0,
        )  # [B,1,1,F,T]

        out = out.squeeze(1).squeeze(1)          # [B,F,T]
        out = out.permute(0, 2, 1).contiguous()  # [B,T,F]
        return out


# =========================================================
# 4. вариант B: свёртка в домене F×T
#    один и тот же FT-kernel на все каналы
#    и этим же kernel'ом каналы схлопываются
#    [B,C,F,T] -> [B,T,F]
# =========================================================

class TFRToSeqFTPlaneConvCollapse(nn.Module):
    """
    Свёртка в домене частота×время:
    - обучаемый kernel [kF, kT]
    - один и тот же kernel повторяется на все каналы
    - каналы схлопываются внутри conv3d
    - output: [B,T,F]

    Важный смысл:
    weights живут в плоскости F×T, а не в канальной оси.
    """

    def __init__(self, kernel_freq: int = 3, kernel_time: int = 3, bias: bool = True):
        super().__init__()

        if kernel_freq <= 0 or kernel_time <= 0:
            raise ValueError("kernel_freq and kernel_time must be > 0")

        self.kernel_freq = kernel_freq
        self.kernel_time = kernel_time
        self.bias_enabled = bias

        self._built = False
        self._channels = None

        self.ft_kernel = None
        self.ft_bias = None

    def _build(self, channels: int, device, dtype):
        self._channels = channels

        self.ft_kernel = nn.Parameter(
            torch.empty(self.kernel_freq, self.kernel_time, device=device, dtype=dtype)
        )
        nn.init.kaiming_uniform_(
            self.ft_kernel.view(1, 1, self.kernel_freq, self.kernel_time),
            a=math.sqrt(5)
        )

        if self.bias_enabled:
            self.ft_bias = nn.Parameter(
                torch.zeros(1, device=device, dtype=dtype)
            )
        else:
            self.register_parameter("ft_bias", None)

        self._built = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(f"Expected [B,C,F,T], got {tuple(x.shape)}")

        B, C, Freq, Time = x.shape

        if not self._built:
            self._build(C, x.device, x.dtype)

        if C != self._channels:
            raise ValueError(
                f"Channel count changed inside one model instance: expected C={self._channels}, got C={C}"
            )

        # [B,C,F,T] -> [B,1,C,F,T]
        x5 = x.unsqueeze(1)

        # один и тот же FT-kernel повторяем на все каналы
        # kernel: [1,1,C,kF,kT]
        kernel = self.ft_kernel.view(1, 1, 1, self.kernel_freq, self.kernel_time)
        kernel = kernel.repeat(1, 1, C, 1, 1)

        pad_f = self.kernel_freq // 2
        pad_t = self.kernel_time // 2

        out = F.conv3d(
            x5,
            weight=kernel,
            bias=self.ft_bias,
            stride=1,
            padding=(0, pad_f, pad_t),
        )  # [B,1,1,F,T] при kF,kT нечетных

        out = out.squeeze(1).squeeze(1)          # [B,F,T]
        out = out.permute(0, 2, 1).contiguous()  # [B,T,F]
        return out

import torch
import torch.nn as nn


class TFRToSeqPixelWeightCollapse(nn.Module):
    """
    Каждый пиксель (f,t) имеет свой обучаемый вес.

    x: [B, C, F, T]
    w: [F, T]

    y[f,t] = w[f,t] * sum_c x[c,f,t]

    output: [B, T, F]
    """

    def __init__(self):
        super().__init__()
        self.weight = None
        self._built = False
        self._shape = None

    def _build(self, F, T, device, dtype):
        self.weight = nn.Parameter(
            torch.randn(F, T, device=device, dtype=dtype)
        )
        self._shape = (F, T)
        self._built = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        if x.ndim != 4:
            raise ValueError(f"Expected [B,C,F,T], got {tuple(x.shape)}")

        B, C, F, T = x.shape

        if not self._built:
            self._build(F, T, x.device, x.dtype)

        if (F, T) != self._shape:
            raise ValueError(
                f"(F,T) changed inside one model instance: expected {self._shape}, got {(F,T)}"
            )

        # collapse channels
        x = x.sum(dim=1)          # [B, F, T]

        # pixel weighting
        x = x * self.weight       # [B, F, T]

        # transformer layout
        x = x.permute(0, 2, 1)    # [B, T, F]

        return x

In [32]:
# =========================================================
# 7. главная обёртка
# =========================================================

class TFRTransformerWrapper(nn.Module):
    """
    Вход:  [B, C, F, T]
    Выход: [B, num_classes]

    preprocess:
      - "flatten"
      - "channel_conv"
      - "ft_plane_conv"
    """
    def __init__(
        self,
        num_classes: int = 2,
        dropout: float = 0.5,
        *,
        seq_len: int = 1001,
        pooling: str = SeqPool(mode = "softmax"),
        preprocess: str = TFRToSeqFlatten(),
        kernel_freq: int = 3,
        kernel_time: int = 3,
        embed_dim: int = 256,
        nhead: int = 8,
        dim_fc: int = 512,
        num_layers: int = 4,
    ):
        super().__init__()

        self.num_classes = num_classes
        self.pooling = pooling
        self.preprocess = preprocess
        self.transformer = Transformer(
            seq_len=seq_len,
            embed_dim=embed_dim, 
            nhead=nhead, 
            dim_fc=dim_fc, 
            num_layers=num_layers,
            dropout=dropout
        )

        self.token_head = nn.Linear(embed_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        [B,C,F,T] -> [B,num_classes]
        """
        preprocess = self.preprocess(x)
        feat = self.transformer(preprocess)
        pooled = self.pooling(feat)
        # logits = self.token_head(pooled)
        return pooled

In [6]:
with open('../patients.yaml') as f:
    test = yaml.safe_load(f)

In [7]:
pat_config = test['default']

In [8]:
# time_frequency_file = pat_config['time_frequency_file'] 
time_frequency_file = "../specs_with_car/tfr_s11.fif"
min_f, max_f, min_t, max_t = pat_config['min_f'], pat_config['max_f'], pat_config['min_t'], pat_config['max_t']

tfr = mne.time_frequency.read_tfrs(time_frequency_file)[0]

Reading ../specs_with_car/tfr_s11.fif ...


/tmp/ipykernel_2158090/1871218919.py:5: RuntimeWarning: This filename (../specs_with_car/tfr_s11.fif) does not conform to MNE naming conventions. All tfr files should end with -tfr.h5 or _tfr.h5
  tfr = mne.time_frequency.read_tfrs(time_frequency_file)[0]


Not setting metadata


In [9]:
y = np.where(tfr.events[:, 2] == 9, 1, 0)

In [10]:
# X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, min_f:max_f, min_t:max_t]
X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, :-50, :]
del tfr
gc.collect() # Теперь память гарантировано свободна

20

In [11]:
# if preprocess == "flatten":
#     preprocess = TFRToSeqFlatten()

# elif preprocess == "channel_conv":
#     preprocess = TFRToSeqChannelConvCollapse()

# elif preprocess == "ft_plane_conv":
#     preprocess = TFRToSeqFTPlaneConvCollapse(
#         kernel_freq=kernel_freq,
#         kernel_time=kernel_time,
#     )

# elif preprocess == "":
#     preprocess = TFRToSeqPixelWeightCollapse(
#         freq_bins=40, 
#         seq_len=100
#     )

# else:
#     raise ValueError(
#         f"Unknown preprocess={preprocess!r}. "
#         f"Use 'flatten', 'channel_conv', or 'ft_plane_conv'."
#     )

In [75]:
cv=False
max_epochs = 10
in_channels = 7 
num_classes = 2
channels = X.shape[1]
seed = 42
test_size = 0.2
n_trials_optuna_whole = 10
n_startup_trials, n_warmup_steps = 5, 5
patience=6
constraints_func = slope_constraint
cv_aggregate = "median"
sampler = optuna.samplers.NSGAIISampler(
    seed=seed,
    constraints_func=constraints_func,   # <-- сюда
)

study = optuna.create_study(
    # макс метрику качества, мин метрику ошибки
    directions=["maximize", "minimize"],
    # sampler=optuna.samplers.TPESampler(), для сингл обджектива
    sampler=sampler
    # pruner=optuna.pruners.MedianPruner(n_startup_trials=n_startup_trials, n_warmup_steps=n_warmup_steps) # может вызвать ошибки, так как прунер настроен на одиночную метрику
)

/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``constraints_func`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-03-16 21:11:34,877] A new study created in memory with name: no-name-e22c4d9a-0e00-4a0f-82ef-d17e0fd8c70f


In [76]:
@beartype
def params_fn_factory(
    *,
    num_classes: int,
    pooling: SeqPool = SeqPool(mode="softmax"),
    preprocess: TFRToSeqFlatten | TFRToSeqChannelConvCollapse | TFRToSeqFTPlaneConvCollapse | TFRToSeqPixelWeightCollapse = TFRToSeqPixelWeightCollapse(),
    ) -> Callable[[Any], Params]:
    Flatten, ChannelCollapse, PlaneCollapse, PixelWeightCollapse = TFRToSeqFlatten(), TFRToSeqChannelConvCollapse(), TFRToSeqFTPlaneConvCollapse(), TFRToSeqPixelWeightCollapse()
    def _params_fn(trial) -> Params:
        params_dict = {}

        embed_dim = trial.suggest_categorical("embed_dim", [32, 64, 128, 256])
        possible_heads = [h for h in [1, 2, 4, 8] if embed_dim % h == 0]

        params_dict["model"] = {
            "num_classes": num_classes,
            "embed_dim": embed_dim,
            "nhead": trial.suggest_categorical("nhead", possible_heads),
            "dim_fc": trial.suggest_categorical("dim_fc", [64, 128, 256, 512]),
            "num_layers": trial.suggest_int("num_layers", 1, 4),
            "dropout": trial.suggest_float("dropout", 0.1, 0.5),
            "pooling": SeqPool(mode=trial.suggest_categorical("pooling", ["mean", "softmax"])),
            "preprocess": trial.suggest_categorical("preprocess", [Flatten, ChannelCollapse, PlaneCollapse, PixelWeightCollapse]),
        }
        params_dict["optimizer"] = {
                "lr": trial.suggest_float("lr", 1e-5, 3e-3, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            }
        params_dict["tr_dataset"] = {
                "time_crop": None,
                # в будущем:
                # зависимые:
                # "time_crop": (
                #     trial.suggest_int("time_crop", 200, 278)  # или T из ctx, если добавишь ctx
                #     if trial.suggest_categorical("time_crop_on", [0, 1]) else None
                # ),
            }
        params_dict["tr_loader"] = {
                "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
                "shuffle": True
                # можно добавить:
                # "num_workers": 4,
                # "pin_memory": True,
            }
        params_dict["vl_dataset"] = dict(params_dict["tr_dataset"])
        params_dict["vl_loader"] = dict(params_dict["tr_loader"])
        params_dict["vl_loader"]["shuffle"] = False
        return params_dict
    return _params_fn

/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [77]:
objective = make_objective_engine(
    X=X, y=y,
    make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
    run_fold_fn=run_fold_fn_factory(
        ModelCls=TFRTransformerWrapper, device=device, 
        max_epochs=max_epochs, patience=patience,
        TFRDataset=TFRDataset, DataLoader=DataLoader,
        train_one_epoch=train_one_epoch, eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
        loss_metric=loss_slope
    ),
    aggregate_mode=cv_aggregate,
    params_fn=params_fn_factory(
        num_classes=num_classes,
    ),
    objectives_fn=objectives_fn,
    attrs_fn=attrs_fn,
)


In [78]:
study.optimize(objective, n_trials=n_trials_optuna_whole, show_progress_bar=True)

  0%|          | 0/10 [00:00<?, ?it/s]

/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqFlatten() which is of type TFRToSeqFlatten.
  warnings.warn(message)
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqChannelConvCollapse() which is of type TFRToSeqChannelConvCollapse.
  warnings.warn(message)
/trinit

[I 2026-03-16 21:24:26,535] Trial 0 finished with values: [0.6066411238825031, -0.0016541704987034856] and parameters: {'embed_dim': 64, 'nhead': 8, 'dim_fc': 512, 'num_layers': 4, 'dropout': 0.18493564427131048, 'pooling': 'softmax', 'preprocess': TFRToSeqChannelConvCollapse(), 'lr': 0.0003278187653397617, 'weight_decay': 3.6138942712165278e-06, 'batch_size': 64}.


/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqFlatten() which is of type TFRToSeqFlatten.
  warnings.warn(message)
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqChannelConvCollapse() which is of type TFRToSeqChannelConvCollapse.
  warnings.warn(message)
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqFTPlaneConvCollapse() which is of type TFRToSeqFTPlaneConvCollapse.

[I 2026-03-16 21:25:03,309] Trial 1 finished with values: [0.3333333333333333, -0.00013091521603721161] and parameters: {'embed_dim': 32, 'nhead': 2, 'dim_fc': 128, 'num_layers': 1, 'dropout': 0.3736932106048628, 'pooling': 'mean', 'preprocess': TFRToSeqFTPlaneConvCollapse(), 'lr': 0.00043767126303409504, 'weight_decay': 1.7654048052495086e-05, 'batch_size': 32}.


...

[I 2026-03-16 21:47:18,009] Trial 7 finished with values: [0.625668449197861, 0.016535541608736094] and parameters: {'embed_dim': 256, 'nhead': 8, 'dim_fc': 64, 'num_layers': 3, 'dropout': 0.3118602313424026, 'pooling': 'mean', 'preprocess': TFRToSeqChannelConvCollapse(), 'lr': 7.328826823009885e-05, 'weight_decay': 0.0008013508750140629, 'batch_size': 16}.


/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqFlatten() which is of type TFRToSeqFlatten.
  warnings.warn(message)
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqChannelConvCollapse() which is of type TFRToSeqChannelConvCollapse.
  warnings.warn(message)
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqFTPlaneConvCollapse() which is of type TFRToSeqFTPlaneConvCollapse.

[I 2026-03-16 22:00:42,032] Trial 8 finished with values: [0.625668449197861, -0.00026958097111096844] and parameters: {'embed_dim': 256, 'nhead': 8, 'dim_fc': 512, 'num_layers': 3, 'dropout': 0.18970772378422393, 'pooling': 'mean', 'preprocess': TFRToSeqPixelWeightCollapse(), 'lr': 0.000425585549072793, 'weight_decay': 0.00018760068220300674, 'batch_size': 32}.


/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqFlatten() which is of type TFRToSeqFlatten.
  warnings.warn(message)
/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains TFRToSeqChannelConvCollapse() which is of type TFRToSeqChannelConvCollapse.
  warnings.warn(message)
/trinit

[I 2026-03-16 22:01:38,596] Trial 9 finished with values: [0.6111111111111112, 0.0034416888699386767] and parameters: {'embed_dim': 64, 'nhead': 2, 'dim_fc': 256, 'num_layers': 1, 'dropout': 0.35818891836286715, 'pooling': 'softmax', 'preprocess': TFRToSeqFlatten(), 'lr': 0.0019932310438786956, 'weight_decay': 5.161032492347078e-05, 'batch_size': 16}.
